In [1]:
import os
import json
import ast
import re
import requests
from typing import Any, Callable, Dict, List, Optional
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv() # 加载 .env 文件

True

In [3]:
class HelloAgentsLLM:
    """一个极简 LLM 客户端，支持任何兼容 OpenAI 接口的服务。"""

    def __init__(
        self,
        model: Optional[str] = None,
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
        timeout: int = 60,
    ):
        self.model = model or os.getenv("LLM_MODEL_ID")
        api_key = api_key or os.getenv("LLM_API_KEY")
        base_url = base_url or os.getenv("LLM_BASE_URL")

        if not all([self.model, api_key, base_url]):
            raise ValueError("请先在 .env 中配置 LLM_MODEL_ID、LLM_API_KEY、LLM_BASE_URL。")

        self.client = OpenAI(api_key=api_key, base_url=base_url, timeout=timeout)

    def think(
        self,
        messages: List[Dict[str, str]],
        temperature: float = 0,
        stream: bool = False,
    ) -> str:
        """调用模型并返回文本。教学场景下默认 temperature=0，减少随机性。"""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            stream=stream,
        )

        if not stream:
            return response.choices[0].message.content or ""

        chunks = []
        for chunk in response:
            if not chunk.choices:
                continue
            text = chunk.choices[0].delta.content or ""
            print(text, end="", flush=True)
            chunks.append(text)
        print()
        return "".join(chunks)

In [4]:
llm = HelloAgentsLLM()
llm.think([{"role": "user", "content": "用一句话解释什么是智能体。"}])

'智能体（Agent）是指能够感知环境、自主决策并采取行动以实现特定目标的软件或硬件实体，具备自主性、反应性、主动性和交互性等基本特征。'

In [5]:
class ToolExecutor:
    """负责注册、描述和执行工具。"""

    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def register(self, name: str, description: str, func: Callable[[str], str]):
        self.tools[name] = {"description": description, "func": func}
        print(f"✅ 工具已注册: {name}")

    def get(self, name: str) -> Optional[Callable[[str], str]]:
        tool = self.tools.get(name)
        return tool["func"] if tool else None

    def describe(self) -> str:
        return "\n".join(
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        )

    def run(self, name: str, tool_input: str) -> str:
        func = self.get(name)
        if not func:
            return f"工具错误：未找到工具 {name}"
        try:
            return func(tool_input)
        except Exception as e:
            return f"工具执行失败：{type(e).__name__}: {e}"

In [6]:
class DeepXivClient:
    """DeepXiv arXiv API 的极简封装。"""

    BASE_URL = "https://data.rag.ac.cn/arxiv/"

    def __init__(self, token: Optional[str] = None):
        self.token = token or os.getenv("DEEPXIV_API_TOKEN")

    def _get(self, params: Dict[str, Any]) -> Dict[str, Any]:
        headers = {}
        if self.token:
            headers["Authorization"] = f"Bearer {self.token}"

        # 没有 token 时也允许请求：DeepXiv 提供了部分免费测试论文和免费测试 query。
        response = requests.get(self.BASE_URL, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        return response.json()

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        source: str = "arxiv",
        return_contents: bool = False,
        use_fine_rerank: bool = True,
    ) -> Dict[str, Any]:
        return self._get({
            "type": "retrieve",
            "query": query,
            "source": source,
            "top_k": top_k,
            "return_contents": str(return_contents).lower(),
            "use_fine_rerank": str(use_fine_rerank).lower(),
        })

    def brief(self, arxiv_id: str) -> Dict[str, Any]:
        return self._get({"type": "brief", "arxiv_id": arxiv_id})

    def preview(self, arxiv_id: str, characters: int = 6000) -> Dict[str, Any]:
        return self._get({
            "type": "preview",
            "arxiv_id": arxiv_id,
            "characters": characters,
        })

    def raw(self, arxiv_id: str) -> Dict[str, Any]:
        return self._get({"type": "raw", "arxiv_id": arxiv_id})

In [7]:
def format_papers(results: Dict[str, Any], max_items: int = 5) -> str:
    """把 DeepXiv 检索结果压缩成适合喂给 LLM 的文本。"""
    papers = results.get("result", [])[:max_items]
    if not papers:
        return "没有检索到相关论文。"

    lines = []
    for i, paper in enumerate(papers, start=1):
        paper_id = paper.get("arxiv_id") or paper.get("biorxiv_id") or paper.get("medrxiv_id")
        title = paper.get("title", "无标题")
        tldr = paper.get("tldr") or paper.get("abstract", "无摘要")
        date = paper.get("date") or paper.get("publish_at", "未知日期")
        url = paper.get("url") or paper.get("src_url", "")
        citations = paper.get("citation_count", paper.get("citations", "未知"))

        lines.append(
            f"[{i}] {title}\n"
            f"ID: {paper_id}\n"
            f"Date: {date}\n"
            f"Citations: {citations}\n"
            f"URL: {url}\n"
            f"Summary: {tldr}"
        )
    return "\n\n".join(lines)


deepxiv = DeepXivClient()


def deepxiv_search_tool(query: str) -> str:
    """供 Agent 调用的论文搜索工具。"""
    print(f"🔍 DeepXivSearch: {query}")
    results = deepxiv.retrieve(query=query, top_k=5, return_contents=False)
    return format_papers(results, max_items=5)

In [8]:
tools = ToolExecutor()
tools.register(
    "DeepXivSearch",
    "搜索 arXiv / bioRxiv / medRxiv 论文。当你需要查找某个研究方向、方法或论文证据时使用。",
    deepxiv_search_tool,
)

✅ 工具已注册: DeepXivSearch


In [9]:
REACT_SYSTEM_PROMPT = """
你是一个能够调用外部工具的研究智能体。
你需要在每一步输出一个 JSON 对象，不要输出 Markdown，不要输出额外解释。

可用工具:
{tools}

输出格式:
{{
  "thought": "你对当前问题的分析",
  "action": {{
    "name": "工具名称，或 Finish",
    "input": "工具输入，或最终答案"
  }}
}}

规则:
1. 如果需要查论文或外部知识，调用 DeepXivSearch。
2. 如果已经有足够证据回答问题，使用 Finish。
3. 不要编造检索结果中没有的信息。
"""


def safe_json_loads(text: str) -> Dict[str, Any]:
    """尽量从模型输出中提取 JSON。"""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if not match:
            raise
        return json.loads(match.group(0))


class ReActAgent:
    def __init__(self, llm: HelloAgentsLLM, tools: ToolExecutor, max_steps: int = 4):
        self.llm = llm
        self.tools = tools
        self.max_steps = max_steps

    def run(self, question: str) -> str:
        history = []

        for step in range(1, self.max_steps + 1):
            print(f"\n--- ReAct 第 {step} 步 ---")
            messages = [
                {
                    "role": "system",
                    "content": REACT_SYSTEM_PROMPT.format(tools=self.tools.describe()),
                },
                {
                    "role": "user",
                    "content": (
                        f"问题：{question}\n\n"
                        f"历史观察：\n{chr(10).join(history) if history else '暂无'}"
                    ),
                },
            ]

            raw = self.llm.think(messages)
            decision = safe_json_loads(raw)

            thought = decision.get("thought", "")
            action = decision.get("action", {})
            name = action.get("name")
            tool_input = action.get("input", "")

            print(f"🧠 Thought: {thought}")
            print(f"🎬 Action: {name}[{tool_input}]")

            if name == "Finish":
                print("✅ ReAct 完成。")
                return tool_input

            observation = self.tools.run(name, tool_input)
            print(f"👀 Observation:\n{observation[:1200]}")

            history.append(f"Action: {name}[{tool_input}]")
            history.append(f"Observation: {observation}")

        return "已达到最大步数，但还没有得到最终答案。"

In [10]:
agent = ReActAgent(llm, tools)
result = agent.run("请帮我找几篇关于 large language model evaluation 的代表性论文。")
print(result)


--- ReAct 第 1 步 ---
🧠 Thought: 用户需要关于 large language model evaluation 的代表性论文。我应该使用 DeepXivSearch 工具搜索相关主题的高质量论文。
🎬 Action: DeepXivSearch[large language model evaluation]
🔍 DeepXivSearch: large language model evaluation
👀 Observation:
[1] A Survey on Evaluation of Large Language Models
ID: 2307.03109
Date: 2023-07-06T16:28:35Z
Citations: 3017
URL: https://arxiv.org/pdf/2307.03109v9
Summary: This survey reviews LLM evaluation methods across tasks, benchmarks, and challenges, offering a systematic taxonomy and highlighting key trends and future directions in the rapidly evolving field of LLM assessment.

[2] Evalet: Evaluating Large Language Models through Functional Fragmentation
ID: 2509.11206
Date: 2025-09-14
Citations: 1
URL: https://arxiv.org/abs/2509.11206v4
Summary: This paper addresses the limitations of holistic LLM evaluation by proposing functional fragmentation to enable fine-grained analysis of model outputs, contributing an interactive system (Evalet) that helps practitione

In [12]:
PLANNER_PROMPT = """
你是一个严谨的任务规划专家。请把用户问题拆成 3 到 6 个可执行步骤。
只输出 JSON 列表，不要输出额外文字。

用户问题:
{question}
"""

EXECUTOR_PROMPT = """
你是一个执行专家。请严格根据当前步骤完成任务。

原始问题:
{question}

完整计划:
{plan}

已完成步骤:
{history}

当前步骤:
{current_step}

请只输出当前步骤的结果，不要重复计划。
"""


class Planner:
    def __init__(self, llm: HelloAgentsLLM):
        self.llm = llm

    def plan(self, question: str) -> List[str]:
        raw = self.llm.think([{"role": "user", "content": PLANNER_PROMPT.format(question=question)}])
        try:
            plan = json.loads(raw)
        except json.JSONDecodeError:
            match = re.search(r"\[.*\]", raw, flags=re.S)
            plan = json.loads(match.group(0)) if match else []
        return plan if isinstance(plan, list) else []


class Executor:
    def __init__(self, llm: HelloAgentsLLM):
        self.llm = llm

    def execute(self, question: str, plan: List[str]) -> str:
        history = []
        last_result = ""

        for i, step in enumerate(plan, start=1):
            print(f"\n-> 执行步骤 {i}/{len(plan)}: {step}")
            prompt = EXECUTOR_PROMPT.format(
                question=question,
                plan=json.dumps(plan, ensure_ascii=False),
                history="\n".join(history) if history else "暂无",
                current_step=step,
            )
            last_result = self.llm.think([{"role": "user", "content": prompt}])
            print(f"✅ 结果: {last_result[:500]}")
            history.append(f"步骤 {i}: {step}\n结果: {last_result}")

        return last_result


class PlanAndSolveAgent:
    def __init__(self, llm: HelloAgentsLLM):
        self.planner = Planner(llm)
        self.executor = Executor(llm)

    def run(self, question: str) -> str:
        print("--- 正在生成计划 ---")
        plan = self.planner.plan(question)
        print(json.dumps(plan, ensure_ascii=False, indent=2))

        if not plan:
            return "计划生成失败。"

        print("\n--- 正在执行计划 ---")
        return self.executor.execute(question, plan)

In [13]:
agent = PlanAndSolveAgent(llm)
result = agent.run("请设计一个学习 Agent 的两小时课堂练习流程。")
print(result)

--- 正在生成计划 ---
[
  "明确学习目标：确定本节课希望学生掌握的Agent核心概念（如感知-决策-行动循环、环境交互、目标导向性）。",
  "设计分阶段活动：规划热身（15分钟）、概念讲解与示例分析（30分钟）、小组协作建模练习（45分钟）、成果展示与反馈（20分钟）、总结反思（10分钟）。",
  "开发配套材料：准备Agent案例卡片（如扫地机器人、推荐系统）、空白Agent结构图模板、评估量表（含目标一致性、环境建模、反馈机制等维度）。",
  "设定具体任务指令：为小组分配不同场景（如校园导览Agent、自习室预约Agent），要求绘制其感知输入、内部状态、决策规则、行动输出及环境反馈路径。",
  "嵌入形成性评估：在小组练习中设置3个检查点（如5分钟/15分钟/30分钟），由教师或助教进行快速诊断并提供针对性提示。",
  "规划教师支持策略：准备常见误区应答清单（如混淆Agent与普通程序、忽略环境动态性），并预设2个延伸挑战问题供学有余力者探索。"
]

--- 正在执行计划 ---

-> 执行步骤 1/6: 明确学习目标：确定本节课希望学生掌握的Agent核心概念（如感知-决策-行动循环、环境交互、目标导向性）。
✅ 结果: 学习目标：  
1. 理解Agent的本质定义——一个能持续感知环境、基于内部状态与目标进行自主决策、并执行行动以影响环境的实体；  
2. 掌握Agent的核心运作机制：感知（输入）→ 内部状态更新与目标评估 → 决策（规则/策略选择）→ 行动（输出）→ 环境反馈 → 循环迭代；  
3. 辨析Agent的关键特征：目标导向性（有明确目标驱动行为）、环境交互性（双向感知与作用，非单向响应）、自主性（在约束内独立决策）、反应性与主动性并存；  
4. 能识别真实系统中Agent的体现（如扫地机器人、课程推荐系统），并初步区分Agent与传统程序（如无状态脚本、无反馈闭环的批处理任务）。

-> 执行步骤 2/6: 设计分阶段活动：规划热身（15分钟）、概念讲解与示例分析（30分钟）、小组协作建模练习（45分钟）、成果展示与反馈（20分钟）、总结反思（10分钟）。
✅ 结果: 分阶段活动设计：  
- **热身（15分钟）**：  
  “Agent识别挑战”快问快答+情境投票。教师展示5个真实系统片段（如：自

In [14]:
DRAFT_PROMPT = """
你是一名研究助教。请完成下面任务，要求结构清晰，语言简洁。

任务:
{task}
"""

REFLECT_PROMPT = """
你是一名严格的评审员。请审查下面的初稿，只关注三个问题：
1. 是否回答了任务？
2. 是否存在事实跳跃或证据不足？
3. 结构是否清晰？

任务:
{task}

初稿:
{draft}

请输出具体、可执行的修改建议。如果无需修改，请输出“无需改进”。
"""

REFINE_PROMPT = """
你是一名研究助教。请根据评审意见修改初稿。

任务:
{task}

初稿:
{draft}

评审意见:
{feedback}

请输出修改后的最终版本。
"""


class ReflectionAgent:
    def __init__(self, llm: HelloAgentsLLM, max_iterations: int = 2):
        self.llm = llm
        self.max_iterations = max_iterations
        self.memory: List[Dict[str, str]] = []

    def run(self, task: str) -> str:
        print("--- 初始执行 ---")
        draft = self.llm.think([{"role": "user", "content": DRAFT_PROMPT.format(task=task)}])
        self.memory.append({"type": "draft", "content": draft})

        for i in range(1, self.max_iterations + 1):
            print(f"\n--- 第 {i} 轮反思 ---")
            feedback = self.llm.think([{
                "role": "user",
                "content": REFLECT_PROMPT.format(task=task, draft=draft),
            }])
            self.memory.append({"type": "feedback", "content": feedback})
            print(f"🧾 Feedback: {feedback[:600]}")

            if "无需改进" in feedback:
                break

            print("\n--- 根据反馈优化 ---")
            draft = self.llm.think([{
                "role": "user",
                "content": REFINE_PROMPT.format(
                    task=task,
                    draft=draft,
                    feedback=feedback,
                ),
            }])
            self.memory.append({"type": "draft", "content": draft})

        return draft

In [15]:
agent = ReflectionAgent(llm)
result = agent.run("解释 ReAct 和 Plan-and-Solve 的区别。")
print(result)

--- 初始执行 ---

--- 第 1 轮反思 ---
🧾 Feedback: 无需改进
ReAct 与 Plan-and-Solve 的核心区别如下：

| 维度         | ReAct（Reasoning + Acting）                     | Plan-and-Solve                                 |
|--------------|-----------------------------------------------|----------------------------------------------|
| **核心思想** | 将推理（Reasoning）与工具调用（Acting）交替交织，边想边做。 | 先完成完整、显式的多步规划（Plan），再依序执行求解（Solve）。 |
| **流程结构** | 循环式：`Thought → Action → Observation → Thought → …`（动态迭代） | 两阶段：① 规划阶段（生成步骤化计划）→ ② 执行阶段（按计划逐步求解） |
| **规划粒度** | 隐式、局部、短视：每步仅规划下一步动作，依赖当前观察更新。 | 显式、全局、前瞻性：提前规划全部关键步骤（如“先提取公式，再代入数值，最后计算”）。 |
| **工具使用** | 紧耦合：Action 直接调用外部工具（如计算器、检索API），Observation 即时反馈驱动后续推理。 | 松耦合：规划中可预设工具使用点，但执行阶段才实际调用；工具调用常被封装在求解步骤中。 |
| **典型场景** | 开放域问答、实时信息检索、需试错/验证的任务（如调试、探索性推理）。 | 结构化数学推理、逻辑推导、步骤明确的符号计算任务（如多步代数题）。 |

✅ 简记：  
- **ReAct = 推理与行动“交错滚动”**（Think-Act-See-Think…）  
- **Plan-and-Solve = 先画蓝图，再照图施工**（Plan once → Solve stepwise）  

二者均提升LLM的复杂推理能力，但设计哲学不同：ReAct 强调适应性与交互性，Plan-and-Solve 强调可控性与可解释性。


In [16]:
PLANNING_PROMPT = """
你是一个资深的研究分析师。请将下面的宏观研究问题拆解为 3-5 个具体的、可执行的搜索子问题。
要求：子问题必须具体，能够通过检索论文直接找到答案，不要包含模糊的描述。

原始研究问题: {question}
"""

SYNTHESIZE_PROMPT = """
你是一个专业的学术写手。请根据下面的用户问题和检索到的参考材料，撰写一份结构清晰、证据确凿的研究综述。
请务必引用参考材料中的具体论文标题或观点来支持你的论述。

用户问题: {question}

参考材料（检索结果）:
{context}
"""

class MiniDeepResearchAgent:
    def __init__(self, llm: HelloAgentsLLM, deepxiv_client: DeepXivClient):
        self.llm = llm
        self.deepxiv = deepxiv_client
        self.search_tool = deepxiv_search_tool 
        self.conversation_memory = []

    def run(self, question: str) -> str:
        print(" 开始深度研究任务")
        print(f" 研究问题: {question}\n")

        print("1. 正在生成研究计划...")
        plan_prompt = PLANNING_PROMPT.format(question=question)
        raw_plan = self.llm.think([{"role": "user", "content": plan_prompt}])
        search_steps = [step.strip("0123456789. ") for step in raw_plan.strip().split('\n') if step.strip()]
        
        print(f" 拆解出的检索步骤: {search_steps}\n")
        self.conversation_memory.append(f"研究计划: {search_steps}")

        print("2. 正在执行检索...")
        all_papers = ""
        for i, sub_query in enumerate(search_steps):
            print(f"    执行检索 [{i+1}/{len(search_steps)}]: {sub_query}")
            paper_results = self.search_tool(sub_query) 
            all_papers += f"\n\n--- 基于子问题的检索结果: {sub_query} ---\n{paper_results}"
            print(f"    获取到 {paper_results.count('[1]')} 篇相关论文摘要\n")
        
        self.conversation_memory.append(f"检索证据: {all_papers}")

        print("3. 正在撰写初稿...")
        synthesis_prompt = SYNTHESIZE_PROMPT.format(question=question, context=all_papers)
        draft = self.llm.think([{"role": "user", "content": synthesis_prompt}])
        print(f" 初稿完成 (前 300 字预览): {draft[:300]}...\n")
        self.conversation_memory.append(f"初稿: {draft}")

        print("4. 正在进行质量反思与优化...")
        feedback_prompt = REFLECT_PROMPT.format(task=question, draft=draft)
        feedback = self.llm.think([{"role": "user", "content": feedback_prompt}])
        print(f"   🧾 评审意见: {feedback}")
        
        if "无需改进" not in feedback:
            print("    正在根据反馈修改...")
            final_draft = self.llm.think([
                {"role": "user", "content": REFINE_PROMPT.format(task=question, draft=draft, feedback=feedback)}
            ])
        else:
            final_draft = draft

        print("\n 深度研究任务完成！")
        return final_draft



In [ ]:
llm = HelloAgentsLLM()
deepxiv_client = DeepXivClient()
research_agent = MiniDeepResearchAgent(llm, deepxiv_client)
report = research_agent.run("Large Language Model 在医疗诊断领域的最新研究进展和主要挑战是什么？")
print("\n最终报告:\n", report)